# CNN — Street View House Numbers (SVHN)
## Neural networks for energy applications — Day 4, Exercise 2

In this exercise we train a CNN to classify digits in the **Street View House Numbers**
(SVHN) dataset. SVHN contains real-world images of house numbers captured by Google
Street View cameras. Each image is 32x32 pixels with 3 colour channels and belongs
to one of 10 digit classes (0–9). We will also explore data augmentation techniques
and evaluate the model with a confusion matrix.

**Evaluation protocol.** We hold out 5,000 training images as a **validation set** for
monitoring, keeping the best epoch, the confusion matrix and the augmentation comparison.
The official **test** split (26,032 images) is used once, in Section 7, for the chosen model.

**Runtime.** To fit a laptop CPU, both models train on the same fixed 20,000-image subset of
the training images for 10 epochs.
This takes about 4–6 minutes on a laptop CPU and 1–2 minutes on a GPU or Apple silicon.
Increase `N_TRAIN` (up to 68,257) if you have more time; keep the budget equal for both models.

---
## 0 — Imports and Setup

In [ ]:
import copy
import os

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Use a GPU if there is one (CUDA, or MPS on Apple silicon), otherwise the CPU.
# GPU results can differ slightly from CPU results even with the same seed.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# SVHN is downloaded to ./data unless the COURSE_DATA environment variable names another folder
DATA_ROOT = os.environ.get("COURSE_DATA", "./data")
print(f"Data folder: {DATA_ROOT}")
print("Setup complete.")

---
## 1 — Load SVHN Dataset

### 1.1 Download and Prepare Data

We hold out 5,000 randomly chosen training images for validation. SVHN crops several digits
from the same house-number photograph, and the files do not record which photograph a digit
came from, so neighbouring digits of one sign can land on both sides of a random split:
validation accuracy may be slightly optimistic. The official test split is kept for the end.

In [ ]:
# YOUR CODE HERE
# TODO: Define transform_basic with ToTensor() and
#       Normalize((0.4377, 0.4438, 0.4728), (0.1980, 0.2010, 0.1970))
# TODO: Download SVHN using torchvision.datasets.SVHN(root=DATA_ROOT, split="train", ...)
#       and split="test" for the test set
# TODO: Define classes = [str(i) for i in range(10)]
#       (torchvision stores the digit 0 as label 0, so label k = digit k)
# TODO: Make a fixed random split of the training images with a seeded torch.randperm:
#       5,000 validation indices and N_TRAIN = 20_000 training indices (no overlap);
#       wrap them with Subset(train_dataset, ...)
# TODO: Print number of training/validation/test samples and image shape

### 1.2 Visualize Sample Images

In [ ]:
# YOUR CODE HERE
# TODO: Write imshow_svhn(img) to unnormalize an image tensor
# TODO: Display 16 training images in a 2x8 grid with digit labels

### 1.3 Create DataLoaders

In [ ]:
# YOUR CODE HERE
# TODO: Create train_loader (shuffle=True) and val_loader (shuffle=False) with BATCH_SIZE=128
#       and num_workers=0 (no test loader yet: the test set is used once, in Section 7)
# TODO: Verify the batch shape

---
## 2 — Build CNN for SVHN

We use a similar architecture to our CIFAR-10 model but with three convolutional
blocks and batch normalization from the start, since SVHN is a larger dataset.

Before training we pass a dummy batch through the model and print the output shape of every
convolution, pooling and linear layer: the shape ledger from the slides.

In [ ]:
# YOUR CODE HERE
# TODO: Define class SVHNCNN(nn.Module) with:
#   Block 1: Conv2d(3, 32, 3, pad=1) + BatchNorm2d(32) + ReLU + MaxPool + Dropout2d(0.25)
#   Block 2: Conv2d(32, 64, 3, pad=1) + BatchNorm2d(64) + ReLU + MaxPool + Dropout2d(0.25)
#   Block 3: Conv2d(64, 128, 3, pad=1) + BatchNorm2d(128) + ReLU + MaxPool + Dropout2d(0.25)
#   Classifier: Linear(128*4*4, 256) + ReLU + Dropout(0.5) + Linear(256, 10)
# Hint: after 3 pooling layers, 32 -> 16 -> 8 -> 4
# TODO: Instantiate on device and print parameter count
# TODO: Shape check: register a forward hook on every Conv2d, MaxPool2d and Linear layer
#       (model.named_children()) that prints the layer name and output shape, pass a dummy
#       batch torch.zeros(8, 3, 32, 32) through the model in eval mode under torch.no_grad(),
#       then remove the hooks


---
## 3 — Train and Evaluate

### 3.1 Training Utilities

In [ ]:
# YOUR CODE HERE
# TODO: Define train_one_epoch(model, loader, criterion, optimizer, device)
#       - model.train(), loop batches, zero_grad, forward, loss, backward, step
#       - Return (avg_loss, accuracy)
#
# TODO: Define evaluate(model, loader, criterion, device)
#       - model.eval(), torch.no_grad(), loop batches
#       - Return (avg_loss, accuracy)


### 3.2 Train for 10 Epochs

After every epoch we evaluate on the **validation** set and keep a copy of the weights from
the epoch with the highest validation accuracy.

In [ ]:
# YOUR CODE HERE
# TODO: Define criterion (CrossEntropyLoss) and optimizer (Adam, lr=1e-3)
# TODO: Train for NUM_EPOCHS=10 (classroom budget; with more time, raise it together with N_TRAIN),
#       evaluating on val_loader after each epoch (NOT the test set);
#       store train/val loss and accuracy in a history dict
# TODO: Keep best_state = copy.deepcopy(model.state_dict()) for the best validation accuracy
#       and reload it after training
# TODO: Print epoch stats each iteration

---
## 4 — Plot Training Curves

In [ ]:
# YOUR CODE HERE
# TODO: Create a 1x2 subplot figure:
#       - Left: training and validation loss vs epoch
#       - Right: training and validation accuracy vs epoch, with the kept epoch marked

---
## 5 — Confusion Matrix (validation set)

In [ ]:
# YOUR CODE HERE
# TODO: Collect all predictions and labels from the VALIDATION set
# TODO: Compute the confusion matrix using sklearn.metrics.confusion_matrix
# TODO: Display it using ConfusionMatrixDisplay with cmap="Blues"
# TODO: Print per-class accuracy from the diagonal of the confusion matrix
# Hint: per_class_acc = cm.diagonal() / cm.sum(axis=1)

---
## 6 — Data Augmentation

We train a fresh model on the **same training images** with random augmentations and
compare it with the first model on the **validation** set, with the same epoch budget.

- **Small random crops** (pad by 2 pixels, crop back to 32 × 32): the digit shifts by up to
  2 pixels. SVHN crops often show neighbouring digits at the sides and the label is the digit in
  the centre, so we keep the shifts small enough that the labelled digit stays in the centre.
- **Colour jitter** (brightness, contrast): house numbers appear in many colours and lighting
  conditions.
- **No horizontal flip.** Most digits are not left-right symmetric: a mirrored 2, 3, 4, 5, 6, 7
  or 9 is no longer that digit, so a flip would train the model on images whose label no longer
  matches what they show. An augmentation must preserve the label.

Augmentation is applied to training images only. Validation and test images keep the
deterministic `transform_basic`. Augmented images are harder to fit, so with a short training
budget augmentation can lower validation accuracy even where it would help with longer training:
judge it on validation data rather than assuming a gain.

In [ ]:
# YOUR CODE HERE
# TODO: Define transform_augmented with:
#       RandomCrop(32, padding=2), ColorJitter(brightness=0.2, contrast=0.2), ToTensor(), Normalize()
#       (no horizontal flip: a mirrored digit is not the same digit)
# TODO: Create a second SVHN training dataset with the augmented transforms and a DataLoader
#       over the SAME train_indices (Subset); validation keeps transform_basic
# TODO: torch.manual_seed(RANDOM_STATE), then instantiate a fresh SVHNCNN model, criterion, optimizer
# TODO: Train for the same NUM_EPOCHS, evaluating on val_loader; keep the best validation weights

In [ ]:
# YOUR CODE HERE
# TODO: Create a 1x2 subplot comparing validation accuracy and validation loss
#       for the first model (no augmentation) vs the augmented model
# TODO: Print the best validation accuracy for each
# TODO: Choose final_model and final_name: the model with the higher best validation accuracy

---
## 7 — Final Test Evaluation (once)

The choice is now locked: the model with the higher validation accuracy, with the weights of
its best validation epoch. We evaluate it once on the official test split. Changing the model
after seeing this number would turn the test set into another validation set.

In [ ]:
# YOUR CODE HERE
# TODO: Create test_loader (shuffle=False, num_workers=0)
# TODO: Call evaluate() ONCE on the test set with final_model and print the test loss and accuracy

---
## Summary

In this exercise we:

1. **Loaded and explored** the SVHN dataset (32x32 colour images of house number digits) and
   held out 5,000 training images for validation.
2. **Built a CNN** with three convolutional blocks, batch normalization, and dropout, and
   printed the output shape of every convolution, pooling and linear layer.
3. **Trained** the model for 10 epochs on a 20,000-image subset, plotted training and
   validation curves and kept the best validation epoch.
4. **Computed a confusion matrix** on the validation set to see which digits are confused.
5. **Applied label-preserving data augmentation** (small random crops and colour jitter, no
   horizontal flip) and compared it with the first model on the validation set.
6. **Evaluated the chosen model once** on the test set.

Key takeaways:
- SVHN is larger than CIFAR-10, and its street-level crops vary widely in quality and often
  show neighbouring digits.
- The confusion matrix reveals which digit pairs are commonly misclassified.
- An augmentation must preserve the label: shifts and colour changes keep a digit's identity,
  mirroring does not.
- Compare augmentation choices on validation data with the same budget; use the test set once.